# ML code

In [36]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression

# Column name mapping

In [37]:
def mapping_column_name(df, case='any', mapping_df=None, verbosity=1):
    if mapping_df is None:
        default_column_name_mapping_file_path = 'column_name_mapping.csv'
        try:
            mapping_df=pd.read_csv(default_column_name_mapping_file_path)
        except:
            raise Exception(f'The default column name file for mapping "{default_column_name_mapping_file_path}" was not found')

    mapping_df = mapping_df.query('use_case == @case')
    if len(mapping_df) == 0:
        raise Exception(f'The mapping dataframe does not contains the case: {case}')
    
    df_mapped = df.copy()
    columns_to_map = df.columns 
    if len(columns_to_map) != len(set(columns_to_map)):
        unique_columns = set(columns_to_map)
        for item in unique_columns:
            columns_to_map.to_list().remove(item)
        raise Exception(f'The column names should be unique but duplicate(s) are found: {set(columns_to_map)}')

    mapping_dictionary = dict()
    for col in columns_to_map:
        mask = mapping_df.apply(lambda row: row.astype(str).str.fullmatch(col, case=False, na=False).any(), axis=1)
        mapping_found = mapping_df[mask]
        if len(mapping_found) == 1:
            df_mapped.rename(columns={col : mapping_found['standard_name'].astype(str).iloc[0]}, inplace=True)
            if verbosity > 0: print(f'The column {col} was renamed to {mapping_found["standard_name"].iloc[0]}')
            mapping_dictionary[col] = mapping_found['standard_name'].astype(str).iloc[0]
        elif len(mapping_found) > 1:
            raise Exception(f'The column {col} has multiple mapping toward the following standard name: {mapping_found["standard_name"].to_list()}')
  
    return df_mapped, mapping_dictionary

df = pd.DataFrame([[2, 2.1, 2.2], [3, 3.1, 3.2]], columns=['Cycle', 'TiCl4_Pulse', 'TiCl4_Purge'])
df, _ = mapping_column_name(df,)
df

The column Cycle was renamed to cycle
The column TiCl4_Pulse was renamed to ticl4_pulse_time
The column TiCl4_Purge was renamed to ticl4_purge_time


,cycle,ticl4_pulse_time,ticl4_purge_time
0,2,2.1,2.2
1,3,3.1,3.2


# Create polynomial features for multiple layer process